# Loinc-Type.py

This notebook is used to add the LOINC Lab types (`Order`, `Observation`, or `Both`) for the various LOINC Codes and their embeddings, to the existing embedding files, generated using the embedding notebook, stored in Azure BLOB Storage (for DIBBS-TTC).


## Setup

Make sure that once the compute instance is running, you activate the kernel associated with the DIBBs Env in the upper right dropdown. Its packages are correctly optimized for this notebook and avoids some `numpy` instabilities plaguing Azure.

In [ ]:
pip install azure-keyvault-secrets azure-identity azure-ai-ml azureml-fsspec

Now we'll do our basic, standard authentication work. We need all these variables to be able to access our container storage from a file mount. The `DATASTORE_NAME` is not a protected secret and therefore doesn't need to be stashed in KeyVault, as it's a standard Azure default.

In [ ]:
# Authenticate to Key Vault
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

credential = DefaultAzureCredential()
key_vault = "dibbsttc6059789213"
secret_client = SecretClient(
    vault_url=f"https://{key_vault}.vault.azure.net/", credential=credential
)

SUBSCRIPTION = secret_client.get_secret("subscription").value
RESOURCE_GROUP = secret_client.get_secret("resource-group").value
WS_NAME = secret_client.get_secret("workspace-name").value
DATASTORE_NAME = "workspaceblobstore"

_IMPORTANT_: This step can't be skipped, even though we're not directly using any of the `ml_client` functionality. This authentication and connection step allows us to use this notebook cleanly within our compute ecosystem. Basically, instantiating the class object acts as a connection that allows us to do everything that follows.

In [ ]:
from azure.ai.ml import MLClient

# Authenticate and connect to workspace
ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION,
    RESOURCE_GROUP,
    WS_NAME,
)

Finally, we'll set the variable that points to the LOINC Extract file which contains the LOINC Codes and Loinc Lab Types.

In [ ]:
# The name of the file in blob storage used to add the LOINC types to embeddings
#  Ensure the file exists in the Azure Blob Storage ahead of time
SNOINC_CODE_TYPE_FILE = "./loinc_lab_names_20251107.csv"

In [ ]:
import torch

USE_EXACT_SEARCH = False

if USE_EXACT_SEARCH:
    assert torch.cuda.is_available()

## Step 1: Create File Mount

The Azure Machine Learning File Mount system, though cumbersomely named, allows us to _directly_ access files and objects we have stored in the DIBBs TTC container. Any `.txt` or `.csv` files need to be created as Data Assets (see sidebar on left), while embedding tensor files and any HNSW `.index` files do not, and can simply be loaded directly from storage.

In [ ]:
# Load up the validation set data
from azureml.fsspec import AzureMachineLearningFileSystem

# Instantiate a file system over the workspace so we can interact with data
# assets directly--we get all the goodies like open, ls, etc.
fs = AzureMachineLearningFileSystem(
    f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}"
)

## Step 2: Unpickle Embeddings Files & Add Loinc Type - Then Re-Pickle

Using our mounted file system, we can directly open the embedding files and unpickle them. Remember, each embedding file is stored as a dictionary of not just the embeddings computed by the `sentence-transformers` model, but the standard LOINC codes associated with those embeddings. Our goal is to add the LOINC Lab Type to the model along side the embeddings and then to re-pickle the files with the updated new data element.

In [ ]:
import os
import pickle

loinc_dict_list = []

# get the loinc lab codes and types into a dictionary list for use
# later instead of reading through this file for each embedding file
with fs.open(SNOINC_CODE_TYPE_FILE, mode="r", encoding="utf-8") as file:
    for i, row in enumerate(file):
        # Blob storage is bytes-based, so we need to decode before string operations
        decoded_row = row.decode("utf-8")
        dict_row = {}
        elements = []
        if i > 0:
            elements = decoded_row.strip().split("|")
            dict_row["display_name"] = elements[4]
            dict_row["long_name"] = elements[3]
            dict_row["short_name"] = elements[2]
            dict_row["lab_type"] = elements[1]
            loinc_dict_list.append(dict_row)
print("LOINC TYPES LOADED...")
print("START PROCESSING FILES....")
# loop through each embedding file
for file in fs.ls("embeddings"):
    # skip any sub-folders
    if not fs.isfile(file):
        continue
    print(f"OPEN FILE: {file}")
    # open the embedding file and un-pickle it
    # store the elements in arrays
    with fs.open(file) as fp:
        print("UN-PICKLE")
        cache_data = pickle.load(fp)
        name_codes = cache_data["codes"]
        embeddings = cache_data["embeddings"]
        embeddings_list = embeddings.tolist()  # have to convert to list
        # ran out of memory a few times - so reducing it by emptying out variables
        # where possible
        embeddings = []
        # then later you will convert the entire list using the torch.tensor() function
        # see print statements for testing

        # new empty arrays for the loinc lab types (loinc_types), codes, and embeddings
        loinc_types = []
        loinc_codes = []
        loinc_embeddings = []

        # loop through all the loinc name_codes (long common names, short names, display names)
        # and find a match in the loinc dictionary (created above from the
        # Loinc extract file containing the Loinc Lab Types)
        print("GETTING LOINC LAB TYPES...")
        for i, code in enumerate(name_codes):
            if not code.strip() or code.strip() == "":  # skip empty codes
                continue
            loinc_type = ""
            for loinc_dict in loinc_dict_list:
                if (
                    loinc_dict.get("long_name") == code
                    or loinc_dict.get("short_name") == code
                    or loinc_dict.get("display_name") == code
                ):
                    loinc_type = loinc_dict.get("lab_type")
                    break
            if loinc_type == "":
                print(f"EMPTY LOINC TYPE: {code}")
            loinc_codes.append(code)
            loinc_embeddings.append(embeddings_list[i])
            # ran out of memory a few times - so reducing it by emptying out variables
            # where possible
            embeddings_list[i] = ""
            loinc_types.append(loinc_type)

        # use the embedding file name to be used to
        # store a local update embedding file which will
        # later be 'put' into the embeddings/refined folder in Azure
        print("ALL LOINC TYPES FOUND...")
        local_file = file.split("/", 1)[1]
        updated_embeddings = torch.tensor(loinc_embeddings)
        # ran out of memory a few times - so reducing it by emptying out variables
        # where possible
        loinc_embeddings = []
    print(f"CODES LEN: {len(loinc_codes)}")
    print(f"TYPES LEN: {len(loinc_types)}")
    print(f"EMBEDDINGS LEN: {len(updated_embeddings)}")
    if len(loinc_codes) != len(loinc_types) or len(loinc_codes) != len(updated_embeddings):
        break
    print(f"WRITE LOCAL FILE: {local_file}")
    # first write contents of the updated embedding file locally
    # then upload them to Azure File System
    with open(local_file, mode="wb") as updated_embedding_file:
        pickle.dump(
            {"codes": loinc_codes, "embeddings": updated_embeddings, "loinc_types": loinc_types},
            updated_embedding_file,
        )
    # this overwrites the existing embedding files in the refined folder
    # with the updated files - removed empty codes and all embeddings having a loinc type
    fs.put(local_file, "/embeddings/refined/", overwrite="MERGE_WITH_OVERWRITE")
    print("UPLOAD NEW FILE")
    # remove the local updated embedding file
    os.remove(local_file)
    print(f"RE-PICKLING COMPLETE FOR: {file}")

print("PROCESS DONE!")